# 作业1：Data Types, Functions and Conditionals

- 姓名：吴卓斯
- 学号：20244080329
- 课程：大数据平台与编程实践B

> 这份 Notebook 按照题目顺序，把每个代码框从头到尾依次运行就可以。


## 先做个身份和环境记录（第一个代码框）

In [1]:
import os, sys, uuid, hashlib, platform, getpass

# ===== 1. 填写学号 =====
STUDENT_ID = 20244080329  # 修改为自己的学号

# ===== 2. 打印 conda 环境 =====
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV") or os.path.basename(sys.prefix)
print(f"[Conda 环境] {CONDA_ENV}")

# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")


[Conda 环境] pycourse
[学号] 20244080329
[机器指纹] 72a4fda4ca0e6d59
[RUN_ID] 20244080329-72a4fda4ca0e6d59


## 一、热身：先写几个简单函数

1. `say_hi()`：不需要参数，调用它就打印 `Hello, world!`；
2. `goatPad(s)`：给它一个字符串，它会在字符串前后各加一个 `goat` 再打印出来；
3. `print_n(s, n)`：给它一个字符串 `s` 和一个整数 `n`，它就把这个字符串打印 `n` 次，每次一行。


**1. 先写 `say_hi`**

In [2]:
def say_hi():
    """不接受参数，打印 Hello, world!"""
    print("Hello, world!")


say_hi()


Hello, world!


**2. 再写 `goatPad`**

In [3]:
def goatPad(s):
    """在字符串 s 的前后各拼接一个 goat 后打印。"""
    print("goat" + s + "goat")


goatPad("hello")
goatPad(" MOUNTAIN ")


goathellogoat
goat MOUNTAIN goat


**3. 最后写 `print_n`**

In [4]:
def print_n(s, n):
    """把字符串 s 打印 n 次，每次单独一行。"""
    for _ in range(n):
        print(s)


print_n("Hello, Hive!", 3)


Hello, Hive!
Hello, Hive!
Hello, Hive!


## 二、欧几里得算法

欧几里得算法又叫辗转相除法，用来求两个整数 $m,n$ 的最大公约数（GCD）。
它的核心递推关系是：$\gcd(m,n)=\gcd(n,\,m\bmod n)$，一直算到余数为 0 就停。


**1. 先把 `gcd` 函数写出来**

In [5]:
def gcd(m, n):
    """使用辗转相除法返回两个整数的最大公约数。"""
    while n != 0:
        m, n = n, m % n
    return m


**2. 用三组数字试试看**

In [6]:
pairs = [(2018, 2019), (1200, 300), (5040, 60)]
for m, n in pairs:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")


gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60


**3. 传负数进去会怎样？来聊一聊**

我们先用上面这个函数，直接拿几个负数来测一测，看看会发生什么：


In [7]:
print("gcd(-2018, 2019) =", gcd(-2018, 2019))
print("gcd(2018, -2019) =", gcd(2018, -2019))
print("gcd(-1200, -300) =", gcd(-1200, -300))


gcd(-2018, 2019) = 1
gcd(2018, -2019) = -1
gcd(-1200, -300) = -300


**来看看现象：**

- 如果两个参数**都是负数**（比如 `gcd(-1200, -300)`），在辗转相除的过程中 `m` 会一直是负数，
  最后返回的就是 **`-300`**。但数学上最大公约数约定是正整数，所以这个结果其实不太规范。
- 如果一个是正数、一个是负数，因为 Python 里取模符号 `%` 的余数符号跟除数（分母）保持一致，
  有时候会“碰巧”算出正数，但这得看具体语言怎么定义取模，既不直观，也没法保证换个语言还一样。
- 所以这个“直接相除”的版本，遇到负数时的行为是**说不准的**，这种结果没有意义——GCD 本来就应该总是非负的。

**怎么改进呢？** 很简单，在进入循环之前先用 `abs()` 取绝对值，这样不管传进来的数是正还是负，都能正确返回一个正的 GCD。


In [8]:
def gcd(m, n):
    """健壮版：先取绝对值，再用辗转相除法求最大公约数。"""
    m, n = abs(m), abs(n)
    while n != 0:
        m, n = n, m % n
    return m


# 改进后对正数、负数都能得到正确的正 GCD
for m, n in [(2018, 2019), (1200, 300), (5040, 60),
             (-2018, 2019), (1200, -300), (-1200, -300)]:
    print(f"gcd({m}, {n}) = {gcd(m, n)}")


gcd(2018, 2019) = 1
gcd(1200, 300) = 300
gcd(5040, 60) = 60
gcd(-2018, 2019) = 1
gcd(1200, -300) = 300
gcd(-1200, -300) = 300


## 三、想办法近似欧拉数 e

自然对数的底数 e 可以写成下面这个级数：

$$e=\sum_{k=0}^{\infty}\frac{1}{k!}=1+\frac{1}{1}+\frac{1}{2!}+\frac{1}{3!}+\cdots$$

这里我们约定 $0!=1$。


**1. `euler_limit`：用极限 $(1+\frac{1}{x})^x$ 来近似 e**

In [9]:
def euler_limit(n):
    """取 x=n，返回 (1+1/n)^n 对 e 的近似值（浮点数）。"""
    return (1 + 1 / n) ** n


for n in [1, 10, 1000, 1_000_000]:
    print(f"n={n:>7d}  ->  {euler_limit(n):.10f}")


n=      1  ->  2.0000000000
n=     10  ->  2.5937424601
n=   1000  ->  2.7169239322
n=1000000  ->  2.7182804691


**2. `euler_infinite_sum`：用前 n 项的和来近似 e**

这里要注意，参数是“从 1 开始数”的：`euler_infinite_sum(1)` 只取公式里的第 1 项（也就是 $k=0$），返回 `1`；
而 `euler_infinite_sum(0)` 一项都不取，返回 `0`。


In [10]:
def euler_infinite_sum(n):
    """返回公式(1)中前 n 项之和对 e 的近似值。"""
    total = 0.0
    factorial = 1          # 0! = 1
    for k in range(n):     # k = 0,1,...,n-1，共 n 项
        if k >= 1:
            factorial *= k  # 此时 factorial = k!
        total += 1.0 / factorial
    return total


print("euler_infinite_sum(0) =", euler_infinite_sum(0))
print("euler_infinite_sum(1) =", euler_infinite_sum(1))
print("euler_infinite_sum(4) =", euler_infinite_sum(4))
print("euler_infinite_sum(20) =", euler_infinite_sum(20))


euler_infinite_sum(0) = 0.0
euler_infinite_sum(1) = 1.0
euler_infinite_sum(4) = 2.6666666666666665
euler_infinite_sum(20) = 2.7182818284590455


**3. `euler_approx`：一项项往里加，直到误差小于 $\epsilon$ 为止**

In [11]:
import math


def euler_approx(epsilon):
    """逐项累加 1/k!，直到与真实 e 的误差不超过 epsilon。"""
    total = 0.0
    factorial = 1
    k = 0
    while abs(math.exp(1) - total) > epsilon:
        if k >= 1:
            factorial *= k
        total += 1.0 / factorial
        k += 1
    return total


for eps in [1e-3, 1e-6, 1e-10]:
    approx = euler_approx(eps)
    print(f"epsilon={eps:g}: 近似={approx:.12f}, 真实={math.exp(1):.12f}, 误差={abs(math.exp(1)-approx):.2e}")


epsilon=0.001: 近似=2.718055555556, 真实=2.718281828459, 误差=2.26e-04
epsilon=1e-06: 近似=2.718281525573, 真实=2.718281828459, 误差=3.03e-07
epsilon=1e-10: 近似=2.718281828447, 真实=2.718281828459, 误差=1.23e-11


**4. `print_euler_sum_table`：把 1 到 n 项的近似值逐行打印出来**

In [12]:
def print_euler_sum_table(n):
    """逐行打印 euler_infinite_sum(k)，k 从 1 到 n。"""
    for k in range(1, n + 1):
        print(euler_infinite_sum(k))


print_euler_sum_table(10)


1.0
2.0
2.5
2.6666666666666665
2.708333333333333
2.7166666666666663
2.7180555555555554
2.7182539682539684
2.71827876984127
2.7182815255731922


## 四、判断整数的一些性质

一个正整数 $n$ 是 2 的幂，当且仅当它能写成 $n=2^p$（$p\ge 0$）。
这道题只允许用除法和取模（`%`），不能用 `math.sqrt`。


**1. `is_power_of_2`：先判断它是不是 2 的幂**

In [13]:
def is_power_of_2(n):
    """若 n 是 2 的幂则返回 True，否则返回 False。"""
    if n < 1:
        return False
    while n > 1:
        if n % 2 != 0:      # 不断除以 2，过程中一旦出现余数即不是 2 的幂
            return False
        n = n // 2
    return True


for x in [1, 2, 3, 4, 8, 10, 16, 64, 100, 1024]:
    print(f"is_power_of_2({x:>4d}) = {is_power_of_2(x)}")


is_power_of_2(   1) = True
is_power_of_2(   2) = True
is_power_of_2(   3) = False
is_power_of_2(   4) = True
is_power_of_2(   8) = True
is_power_of_2(  10) = False
is_power_of_2(  16) = True
is_power_of_2(  64) = True
is_power_of_2( 100) = False
is_power_of_2(1024) = True


**2. `is_power(b, n)`：推广一下，判断 n 是不是 b 的幂**

In [14]:
def is_power(b, n):
    """若 n 是 b 的某次幂（n = b^p, p>=0）则返回 True。"""
    if n == 1:
        return True          # 任意 b 的 0 次幂都是 1
    if b <= 1:
        return n == 1        # 处理 b=1 等退化情形
    while n > 1:
        if n % b != 0:
            return False
        n = n // b
    return n == 1


for b, n in [(2, 16), (2, 10), (3, 81), (3, 80), (10, 1000), (5, 1), (6, 216)]:
    print(f"is_power({b}, {n}) = {is_power(b, n)}")


is_power(2, 16) = True
is_power(2, 10) = False
is_power(3, 81) = True
is_power(3, 80) = False
is_power(10, 1000) = True
is_power(5, 1) = True
is_power(6, 216) = True
